In [1]:
%matplotlib inline

In [2]:
! pip install -q transformers[sentencepiece] fastbook fastai nbdev plum-dispatch evaluate seqeval onnxruntime onnx

In [3]:
!git clone https://github.com/msi1427/blurr.git
%cd blurr

fatal: destination path 'blurr' already exists and is not an empty directory.
/content/blurr


In [4]:
import torch
from transformers import AutoModelForSequenceClassification, AutoConfig
from fastai.text.all import *
from blurr.text.data.all import *
from blurr.text.modeling.all import *

In [5]:
from tqdm.notebook import tqdm
import numpy as np

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
%cd /content/drive/MyDrive/udemy_project/Onnx Inference

/content/drive/MyDrive/udemy_project/Onnx Inference


In [8]:
df = pd.read_csv("/content/drive/MyDrive/udemy_project/Datasets/course_details.csv")
df.head()

,title,url,description,topic,topic_list
0,The Complete Python Bootcamp From Zero to Hero in Python,https://www.udemy.com/course/complete-python-bootcamp,"Become a Python Programmer and learn one of employer's most requested skills of 2023! This is the most comprehensive, yet straight-forward, course for the Python programming language on Udemy! Whether you have never programmed before, already know basic syntax, or want to learn about the advanced features of Python, this course is for you! In this course we will teach you Python 3. With over 100 lectures and more than 21 hours of video this comprehensive course leaves no stone unturned! This course includes quizzes, tests, coding exercises and homework assignments as well as 3 major projec...",Python,"['Development', 'Programming Languages', 'Python']"
1,The Complete Full-Stack Web Development Bootcamp,https://www.udemy.com/course/the-complete-web-development-bootcamp,"Welcome to the Complete Web Development Bootcamp, the only course you need to learn to code and become a full-stack web developer. With 150,000+ ratings and a 4.8 average, my Web Development course is one of the HIGHEST RATED courses in the history of Udemy! At 62+ hours, this Web Development course is without a doubt the most comprehensive web development course available online. Even if you have zero programming experience, this course will take you from beginner to mastery. Here's why: The course is taught by the lead instructor at the App Brewery, London's leading in-person programmin...",Web Development,"['Development', 'Web Development', 'Node.Js', 'MongoDB', 'JavaScript']"
2,100 Days of Code™: The Complete Python Pro Bootcamp,https://www.udemy.com/course/100-days-of-code,"Welcome to the 100 Days of Code - The Complete Python Pro Bootcamp, the only course you need to learn to code with Python. With over 500,000 5 STAR reviews and a 4.8 average, my courses are some of the HIGHEST RATED courses in the history of Udemy! 100 days, 1 hour per day, learn to build 1 project per day, this is how you master Python. At 60+ hours, this Python course is without a doubt the most comprehensive Python course available anywhere online. Even if you have zero programming experience, this course will take you from beginner to professional. Here's why: The course is taught by...",Python,"['Development', 'Programming Languages', 'Python', 'Data Science', 'Flask', 'Web Scraping']"
3,The Web Developer Bootcamp 2026,https://www.udemy.com/course/the-web-developer-bootcamp,"Now with over 10 hours of React content. Massive new React ""expansion pack"" covers: React basics, JSX, props, state, Vite, MaterialUI, hooks, useEffect, React design patterns, and more. Hi! Welcome to the brand new version of The Web Developer Bootcamp, Udemy's most popular web development course. This course was completely overhauled to prepare students for the current job market, now with over 70 hours of total content! This is the only course you need to learn web development. There are a lot of options for online developer training, but this course is without a doubt the most comprehe...",Web Development,"['Development', 'Web Development']"
4,"React - The Complete Guide (incl. Next.js, Redux)",https://www.udemy.com/course/react-the-complete-guide-incl-redux,"This bestselling course by the author of ""React Key Concepts"" has turned more students into ReactJS developers than any other courses - more than 1,000,000 and counting! - Fully updated for React 19! - A Course For Busy Customers & Business Professionals! This course also comes with two paths which you can take: The ""complete"" path (full >40h course) and the ""summary"" (fast-track) path (~4h summary module) - you can choose the path that best fits your time requirements! React.js is THE most popular JavaScript library you can use and learn these days to build modern, reactive user interfa...",React JS,"['Development', 'Programming Languages', 'React JS', 'React Hooks']"


In [9]:
df = df.dropna().reset_index(drop=True)
df.shape

(11057, 5)

In [10]:
topic_lists = df.topic_list.to_list()
topic_count = {}
for topics in topic_lists:
    topic_list = eval(topics)
    for topic in topic_list:
        if topic in topic_count.keys():
            topic_count[topic] += 1
        else:
            topic_count[topic] = 1
print(f"Number of Topics: {len(topic_count)}")
print(topic_count)

threshold = int(len(df) * 0.01)
rare_topics = [cat for cat, count in topic_count.items() if count < threshold]
len(rare_topics)

Number of Topics: 2436
{'Development': 1598, 'Programming Languages': 336, 'Python': 272, 'Web Development': 479, 'Node.Js': 38, 'MongoDB': 15, 'JavaScript': 103, 'Data Science': 243, 'Flask': 9, 'Web Scraping': 10, 'React JS': 61, 'React Hooks': 6, 'Angular': 34, 'Java': 125, 'Machine Learning': 88, 'R (programming language)': 17, 'Software Development Tools': 101, 'Prompt Engineering': 38, 'Software Testing': 160, 'Selenium WebDriver': 28, 'Automation': 50, 'CSS': 68, 'Web Design': 142, 'Responsive Design': 14, 'HTML': 54, 'OpenAI API': 9, 'ChatGPT': 198, 'Database Design & Development': 98, 'SQL': 105, 'MySQL': 33, 'Game Development': 60, 'Game Development Fundamentals': 44, '2D Game Development': 13, 'Unity': 47, 'C# (programming language)': 57, 'Mobile Development': 58, 'Swift': 10, 'Xcode': 4, 'ARKit': 2, 'iOS Development': 17, 'Spring Framework': 30, 'Spring MVC': 9, 'Hibernate': 7, 'Dart (programming language)': 8, 'Google Flutter': 9, 'Front End Web Development': 26, 'Redux Fr

2370

In [11]:
topic_lists = df.topic_list.to_list()
revised_topic_list = []
indices_to_drop = []

for idx, topics in enumerate(topic_lists):
  topic_list = eval(topics)
  revised_topics = []

  for topic in topic_list:
    if topic not in rare_topics:
      revised_topics.append(topic)

  if len(revised_topics) == 0:
    indices_to_drop.append(idx)
  else:
    revised_topic_list.append(revised_topics)

df = df.drop(indices_to_drop).reset_index(drop=True)
df['revised_topics'] = revised_topic_list
df.shape

(11055, 6)

In [12]:
revised_topics_list = df.revised_topics.to_list()
revised_topic_count = {}
for topics in revised_topics_list:
  topic_list = topics
  for topic in topic_list:
    if topic in revised_topic_count.keys():
      revised_topic_count[topic] += 1
    else:
      revised_topic_count[topic] = 1
print(f"Number of Topics: {len(revised_topic_count)}")

Number of Topics: 66


In [13]:
import json

encode_topic_types = { key: idx for idx, (key, value) in enumerate(revised_topic_count.items())}
with open("topic_types_encoded.json", "w") as fp:
  json.dump(encode_topic_types, fp)

In [14]:
# We need this because for multilabel classification all topics have possibility to be present in the predictions
categorical_topic_list = []
revised_topics_list = df.revised_topics.to_list()

for revised_topics in revised_topics_list:
  categorical_list = [0] * len(encode_topic_types)
  for topic in revised_topics:
    topic_type_index = encode_topic_types[topic]
    categorical_list[topic_type_index] = 1
  categorical_topic_list.append(categorical_list)

df['topic_cat_list'] = categorical_topic_list
df.shape

(11055, 7)

In [15]:
labels = list(encode_topic_types.keys())
len(labels), labels[:5]

(66,
 ['Development',
  'Programming Languages',
  'Python',
  'Web Development',
  'Data Science'])

# Data Split

In [16]:
from fastai.text.all import *

In [17]:
splitter = RandomSplitter(valid_pct=0.1, seed=42)
train_ids, valid_ids = splitter(df)
len(train_ids), len(valid_ids)

(9950, 1105)

In [18]:
valid_df = df.loc[valid_ids]
valid_df.head()

,title,url,description,topic,topic_list,revised_topics,topic_cat_list
2382,Leading Self,https://www.udemy.com/course/leading-self,"“To know thyself is the beginning of wisdom.” -Socrates In today's complex, stressful, and ever-changing world, leaders like yourself are facing unprecedented challenges. It's no longer enough to rely on old models and traditional leadership approaches. To truly thrive and lead your team to success, you must first change the way you lead yourself. This course is for leaders who are looking for innovative, novel approaches to tackle ever-growing leadership issues involving time management, people development, and strategic and design thinking. It's also for leaders who want to excel in the...",Leadership,"['Business', 'Management', 'Leadership']","[Business, Management, Leadership]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1500,The Git and GitHub for Beginners,https://www.udemy.com/course/git-github-crash-course,"This course is about basic features of the Git and GitHub. Learn how to use Git and GitHub in about an HOUR! Commits, branches, merging, GitHub and Git Objects If those terms are new to you - jump in and you will learn what is Git and how to use its core features. Become familiar with Git and GitHub to use it in your own projects and projects you potentially will contribute to in the future. This is fast Git and GitHub Crash Course that will give you basic understanding of the Git and GitHub. Also using several practice activities you will practice core Git and GitHub feature like committ...",Git,"['Development', 'Software Development Tools', 'Git']",[Development],"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
10247,Redesign Your Career with a new Personal Business Model,https://www.udemy.com/course/redesign-your-career,"Replace career confusion with career confidence, whether you want to improve in your current position, change jobs, or start your own enterprise. This hands-on, experiential course teaches you step-by-step how to define and redesign your Personal Business Model™ — the logic by which you create and deliver value. Gain clarity and power at work by mastering the systematic, repeatable way to optimize the most important business model of all: business model you. Taught directly by the author of the global bestseller Business Model You.",Career Change,"['Personal Development', 'Career Development', 'Career Change', 'Personal Branding']","[Personal Development, Career Development]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
4597,Getting Started with Cryptohopper,https://www.udemy.com/course/miggity_ch_getting_started,"In this course, I discuss the ins&outs of Cryptohopper! How to utilize this tool to generate a passive income in Cryptocurrency through automated trading. I cover Dashboard Overview, Configs(TA/Signals), Back-testing, Stats/Profit Calculations and Trading Strategy. I do cover strategy but the primary focus of this course is the functionally of Cryptohopper.",Cryptocurrency,"['Finance & Accounting', 'Cryptocurrency & Blockchain', 'Cryptocurrency', 'Bitcoin Trading', 'Investing & Trading', 'Other IT & Software']","[Finance & Accounting, Cryptocurrency & Blockchain, Cryptocurrency, Investing & Trading, Other IT & Software]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
2091,International Logistics & Transportation in Supply Chain.,https://www.udemy.com/c

We will be using `valid_df` for all inference testing

# Fastai & Blurr Inference

In [19]:
model_path = "models/course-classifier-stage-1.pkl"
learner_inf = load_learner(model_path)

In [20]:
learner_inf.blurr_predict("random placeholder")


[{'labels': [],
  'scores': [],
  'class_indices': [0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0],
  'class_labels': ['Development', 'Programming Languages', 'Python', 'Web Development', 'Data Science', 'Java', 'Software Testing', 'Web Design', 'ChatGPT', 'Software Engineering', 'Generative AI (GenAI)', 'Artificial Intelligence (AI)', 'Data Analysis', 'Leadership', 'Financial Trading', 'Finance Fundamentals', 'Business Fundamentals', 'Business Strategy', 'IT Certifications', 'Cybersecurity', 'User Experience (UX) Design', 'IT & Software', 'Management Skills', 'Project Management', 'Cryptocurrency', 'Architectural Design', 'Business', 'Business A

In [21]:
# learner_inf.blurr_predict("Master Adobe Photoshop and graphic design fundamentals for creating stunning visuals")
learner_inf.blurr_predict("random placeholder")[0]['labels']

[]

# Evaluation

In [22]:
from sklearn import metrics

def metric_measures(test_df, preds):

  targets = [np.asarray(target) for target in test_df['topic_cat_list'].to_list()]
  outputs = [np.asarray(pred) for pred in preds]


  accuracy = metrics.accuracy_score(targets, outputs)
  f1_score_micro = metrics.f1_score(targets, outputs, average='micro')
  f1_score_macro = metrics.f1_score(targets, outputs, average='macro')

  print(f"F1 Score (Micro) = {f1_score_micro}")
  print(f"F1 Score (Macro) = {f1_score_macro}")

  return

In [23]:
preds = []
for idx, row in tqdm(valid_df.iterrows(), total=len(valid_df)):
  desc = row['description']
  labels = learner_inf.blurr_predict(desc)[0]['class_indices']
  # pred_genres = [0] * len(encode_genre_types)
  # for label in labels:
  #   pred_genres[encode_genre_types[label]] = 1
  preds.append(labels)

preds[0][:20]

  0%|          | 0/1105 [00:00<?, ?it/s]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [24]:
metric_measures(valid_df, preds)

F1 Score (Micro) = 0.5179171332586786
F1 Score (Macro) = 0.10147291902140158


# Convert to ONNX

### ONNX
ONNX (Open Neural Network Exchange) is a open standard for representing machine learning models.
- It allows developers to move models between different frameworks (such as PyTorch, TensorFlow, and Caffe2) without losing performance or accuracy.
ONNX makes it easier to build and run AI models on a variety of hardware, including GPUs, CPUs, and custom accelerators.
- This standard helps eliminate the need for multiple conversion tools and provides a unified representation of the model across different tools and frameworks.
- ONNX is being developed by a collaboration of companies including Microsoft, Facebook, Amazon, and IBM, among others, making it a well-supported and widely-adopted standard.
- Converting to ONNX runtime often makes model faster.

In [25]:
model_path = "models/course-classifier-stage-1.pkl"
learner_inf = load_learner(model_path)

In [26]:
learner_inf.model.hf_model

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (

In [27]:
!pip install -q onnxscript

In [28]:
# classifier = learner_inf.model.hf_model.eval()

# torch.onnx.export(
#     classifier,
#     torch.LongTensor([[0] * 512]),
#     'models/course-classifier.onnx',
#     verbose=True,
#     input_names=['input_ids'],
#     output_names=['output'],
#     opset_version=14,
#     dynamic_axes={
#         'input_ids': {0: 'batch_size', 1: 'sequence_len'},
#         'output': {0: 'batch_size'}
#     }
# )
classifier = learner_inf.model.hf_model.eval()

torch.onnx.export(
    classifier,
    torch.LongTensor([[0] * 512]),
    'models/course-classifier.onnx',
    verbose=True,
    input_names=['input_ids'],
    output_names=['output'],
    opset_version=14,
    dynamic_axes={
        'input_ids': {0: 'batch_size', 1: 'sequence_len'},
        'output': {0: 'batch_size'}
    },
    dynamo=False
)

In [29]:
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession('models/course-classifier.onnx')
dummy_input = np.zeros((1, 512), dtype=np.int64)
outputs = sess.run(None, {'input_ids': dummy_input})
print(outputs[0].shape)   # should print (1, 66)

(1, 66)


In [30]:
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    'models/course-classifier.onnx',
    'models/course-classifier-quantized.onnx',
    weight_type=QuantType.QUInt8,
)

In [31]:
import os

print(os.path.exists("models/course-classifier.onnx"))
print(os.path.exists("models/course-classifier-quantized.onnx"))

True
True


In [32]:
import onnxruntime as rt

inf_session = rt.InferenceSession(
    "models/course-classifier-quantized.onnx",
    providers=["CPUExecutionProvider"]
)

print("ONNX model loaded successfully!")
print("Inputs:")
for inp in inf_session.get_inputs():
    print(inp.name, inp.shape, inp.type)

print("Outputs:")
for out in inf_session.get_outputs():
    print(out.name, out.shape, out.type)

ONNX model loaded successfully!
Inputs:
input_ids ['batch_size', 'sequence_len'] tensor(int64)
Outputs:
output ['batch_size', 66] tensor(float)


# ONNX Inference

## Normal ONNX

In [38]:
print(learner_inf.model.hf_model.config._name_or_path)

distilroberta-base


In [39]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [40]:
import onnxruntime as ort

inf_session = ort.InferenceSession('models/course-classifier-quantized.onnx')
input_name = inf_session.get_inputs()[0].name
output_name = inf_session.get_outputs()[0].name

print(input_name, output_name)

input_ids output


In [41]:
class_labels = labels  # from: labels = list(encode_topic_types.keys())
print(len(class_labels))  # sanity check -> should print 66

66


In [45]:
class_labels = list(encode_topic_types.keys())

# Sanity checks - run these and confirm output before proceeding
print(len(class_labels))         # should be 66
print(class_labels[:5])          # should show actual topic strings, e.g. ['Development', 'Programming Languages', ...]
print(type(class_labels[0]))     # should be <class 'str'>, NOT <class 'int'>

66
['Development', 'Programming Languages', 'Python', 'Web Development', 'Data Science']
<class 'str'>


In [46]:
preds = []
for idx, row in tqdm(valid_df.iterrows(), total=valid_df.shape[0]):
    desc = row['description']
    input_ids = tokenizer(desc)['input_ids'][:512]

    probs = inf_session.run([output_name], {input_name: [input_ids]})[0]
    probs = torch.FloatTensor(probs)

    masks = torch.sigmoid(probs) >= 0.5
    pred_labels = [class_labels[i] for i, mask in enumerate(masks[0]) if mask]

    pred_topics = [0] * len(encode_topic_types)
    for label in pred_labels:
        pred_topics[encode_topic_types[label]] = 1
    preds.append(pred_topics)

  0%|          | 0/1105 [00:00<?, ?it/s]

In [47]:
metric_measures(valid_df, preds)

F1 Score (Micro) = 0.5138028169014085
F1 Score (Macro) = 0.09825407556025854


## Quantized ONNX

In [48]:
import onnxruntime as rt
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")

class_labels = list(encode_topic_types.keys())

inf_session = rt.InferenceSession('models/course-classifier-quantized.onnx')
input_name = inf_session.get_inputs()[0].name
output_name = inf_session.get_outputs()[0].name

In [49]:
preds = []
for idx, row in tqdm(valid_df.iterrows(), total=valid_df.shape[0]):
  desc = row['description']
  input_ids = tokenizer(desc)['input_ids'][:512]

  probs = inf_session.run([output_name], {input_name: [input_ids]})[0]
  probs = torch.FloatTensor(probs)

  masks = torch.sigmoid(probs) >= 0.5
  labels = [class_labels[idx] for idx, mask in enumerate(masks[0]) if mask]

  pred_topics = [0] * len(encode_topic_types)
  for label in labels:
    pred_topics[encode_topic_types[label]] = 1
  preds.append(pred_topics)

  0%|          | 0/1105 [00:00<?, ?it/s]

In [50]:
metric_measures(valid_df, preds) #

F1 Score (Micro) = 0.5138028169014085
F1 Score (Macro) = 0.09825407556025854
